# Heritage Short: score candidates on Kaggle (ACE-Step 1.5 or MusicGen)
Pushed by `pipeline/music.py` via `pipeline/kaggle_job.py`. The JOBS cell lists the candidates (caption, bpm, key,
time signature, duration, seed). Modes (`config/models.json: music.mode`, see FAILURES.md #16-21):
- `gpu-split`: ACE-Step in float32 with its diffusion transformer dispatched block by block across BOTH T4s
  (the T4 has no bfloat16 and fp16 overflows; fp32 is ~18 GB, more than one T4); other components on GPU 0;
- `gpu-bf16`: ACE-Step in bfloat16 on a GPU that supports it; `gpu-mixed`: fp16 transformer + fp32 VAE;
- `cpu`: ACE-Step fp32 on the CPU; `musicgen`: MusicGen in fp32 on one GPU (tempo and key go in its text prompt).
With `test_first_s`, a short test clip is rendered first and the session stops if it fails.
Writes `out/<cue>_c<n>.wav` and `timing.jsonl` (GPU seconds and peak VRAM per GPU) to /kaggle/working.

In [ ]:
# @JOBS (replaced at push time)
JOBS = {"job_id": "local-test", "config": {}, "jobs": []}

In [ ]:
import json, os, sys, time, glob, hashlib, subprocess, traceback
T0 = time.time()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/out", exist_ok=True)
_log = open(f"{OUT}/timing.jsonl", "a")
def log(**kw):
    """One JSON line per attempt; pipeline/kaggle_job.py copies these into runs.jsonl."""
    _log.write(json.dumps({"job_id": JOBS["job_id"], **kw}) + "\n"); _log.flush(); print(kw)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.39.0", "soundfile"], check=True)

In [ ]:
import numpy as np, soundfile as sf, torch

cfg = JOBS["config"]
MODE = cfg.get("mode", "gpu-split")
DEV = "cpu" if MODE == "cpu" else "cuda"
INFO, LAT = {}, {}
if MODE == "musicgen":
    from transformers import AutoProcessor, MusicgenForConditionalGeneration
    proc = AutoProcessor.from_pretrained(cfg["model"])
    mg = MusicgenForConditionalGeneration.from_pretrained(cfg["model"], torch_dtype=torch.float32).to("cuda").eval()
    RATE = mg.config.audio_encoder.sampling_rate
    def generate(job, seconds):
        inputs = proc(text=[job["caption"]], padding=True, return_tensors="pt").to("cuda")
        torch.manual_seed(job["seed"])
        with torch.no_grad():
            out = mg.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=int(seconds * mg.config.audio_encoder.frame_rate))
        return out[0, 0].cpu().float().numpy()[:, None], RATE
else:
    from diffusers import AceStepPipeline
    DTYPES = {"cpu": torch.float32, "gpu-bf16": torch.bfloat16, "gpu-mixed": torch.float16, "gpu-split": torch.float32}
    if MODE == "gpu-bf16" and not torch.cuda.is_bf16_supported():
        raise SystemExit("no bfloat16 on " + torch.cuda.get_device_name(0))
    pipe = AceStepPipeline.from_pretrained(cfg["model"], torch_dtype=DTYPES[MODE])
    if MODE == "gpu-split":
        from accelerate import dispatch_model, infer_auto_device_map
        from accelerate.hooks import AlignDevicesHook, add_hook_to_module
        if torch.cuda.device_count() < 2:
            raise SystemExit("gpu-split needs two GPUs")
        mods = {n: m for n, m in pipe.components.items() if isinstance(m, torch.nn.Module)}
        big = max(mods, key=lambda n: sum(p.numel() for p in mods[n].parameters()))
        blocks = sorted({type(l[0]).__name__ for l in mods[big].modules() if isinstance(l, torch.nn.ModuleList) and len(l) >= 8})
        dmap = infer_auto_device_map(mods[big], max_memory={0: "9GiB", 1: "13GiB"}, no_split_module_classes=blocks)
        if any(v in ("cpu", "disk") for v in dmap.values()):
            raise SystemExit(f"{big} does not fit on the two GPUs")
        dispatch_model(mods[big], device_map=dmap)
        for n, m in mods.items():
            if n != big:
                m.to("cuda:0")
                add_hook_to_module(m, AlignDevicesHook(execution_device=torch.device("cuda:0"), io_same_device=True))
        INFO = {"split_component": big, "no_split": blocks, "params_b": round(sum(p.numel() for p in mods[big].parameters()) / 1e9, 2),
                "modules_gpu0": sum(v == 0 for v in dmap.values()), "modules_gpu1": sum(v == 1 for v in dmap.values()),
                "components": sorted(mods)}
    elif MODE == "gpu-mixed":
        pipe.vae.to(torch.float32)
        _decode = pipe.vae.decode
        def decode_fp32(z, *a, **k):
            LAT["finite"] = bool(torch.isfinite(z).all())
            return _decode(z.to(torch.float32), *a, **k)
        pipe.vae.decode = decode_fp32
        pipe.enable_model_cpu_offload()
    elif MODE == "gpu-bf16":
        pipe.enable_model_cpu_offload()
    else:
        torch.set_num_threads(os.cpu_count() or 4)
    RATE = pipe.sample_rate
    def generate(job, seconds):
        audio = pipe(prompt=job["caption"], lyrics="[Instrumental]" if job["instrumental"] else "",
                     audio_duration=float(seconds), bpm=job["bpm"], keyscale=job["keyscale"],
                     timesignature=job["timesignature"], num_inference_steps=job["steps"], shift=job["shift"],
                     generator=torch.Generator("cpu").manual_seed(job["seed"])).audios
        return audio[0].T.cpu().float().numpy(), RATE
log(id=None, event="setup", seconds=round(time.time() - T0, 1), mode=MODE, torch=torch.__version__,
    gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [], **INFO)

def peak():
    return [round(torch.cuda.max_memory_allocated(i) / 1e9, 2) for i in range(torch.cuda.device_count())] if DEV == "cuda" else []

def reset():
    for i in range(torch.cuda.device_count() if DEV == "cuda" else 0):
        torch.cuda.reset_peak_memory_stats(i)

In [ ]:
def render(job, seconds, path):
    reset()
    t = time.time()
    wav, rate = generate(job, seconds)
    if not np.isfinite(wav).all() or float(np.abs(wav).max()) < 1e-3:
        raise RuntimeError(f"NaN or silent audio ({MODE}; latents finite: {LAT.get('finite')})")
    sf.write(path, wav, rate)
    return round(time.time() - t, 1), round(len(wav) / rate, 2)

go = True
if cfg.get("test_first_s") and JOBS["jobs"]:
    job = JOBS["jobs"][0]
    try:
        secs, dur = render(job, cfg["test_first_s"], f"{OUT}/out/test.wav")
        log(id="test", status="ok", attempt=1, seed=job["seed"], gpu_seconds=secs, duration_s=dur, file="out/test.wav",
            mode=MODE, peak_vram_gb=peak())
    except BaseException as e:   # a failed test stops the session before spending more GPU time
        log(id="test", status="error", attempt=1, seed=job["seed"], gpu_seconds=0, error=repr(e)[:400], mode=MODE, peak_vram_gb=peak())
        traceback.print_exc()
        go = False

for job in (JOBS["jobs"] if go else []):
    try:
        secs, dur = render(job, job["duration_s"], f"{OUT}/out/{job['id']}.wav")
        log(id=job["id"], status="ok", attempt=1, seed=job["seed"], gpu_seconds=secs if DEV == "cuda" else 0,
            cpu_seconds=secs, duration_s=dur, file=f"out/{job['id']}.wav", mode=MODE, peak_vram_gb=peak())
    except Exception as e:
        log(id=job["id"], status="error", attempt=1, seed=job["seed"], gpu_seconds=0, error=repr(e)[:400], mode=MODE,
            latents_finite=LAT.get("finite"), peak_vram_gb=peak())
        traceback.print_exc()
        if DEV == "cuda":
            torch.cuda.empty_cache()
log(id=None, event="done", seconds=round(time.time() - T0, 1))